In [2]:
import pandas as pd
import numpy as np
import joblib
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from scipy.sparse import hstack

print(" Imports OK")


 Imports OK


In [4]:
DATA_PATH = "labelAMD2.csv"   # your merged dataset
MODEL_OUT = "models/text_cls_balanced_tfidf.joblib"

TEST_SIZE = 0.2
RANDOM_SEED = 42

LABEL_MAP = {
    "HUMAN": 0,
    "MACHINE": 1
}

print(" Config set")


 Config set


In [12]:
DATA_PATH = "labelAMD2.csv"

import os
print("Exists?", os.path.exists(DATA_PATH))


Exists? True


In [13]:
import pandas as pd
df = pd.read_csv(DATA_PATH)
print(len(df))
print(df["label"].value_counts())


121153
label
HUMAN      91269
MACHINE    29884
Name: count, dtype: int64


In [14]:
import re

def normalize_text(t):
    t = t.lower().strip()
    t = re.sub(r"\s+", " ", t)      # normalize whitespace
    t = re.sub(r"[^\w\s']", "", t)  # keep words + apostrophes
    return t

df["text"] = df["text"].astype(str).apply(normalize_text)

# Remove empty after cleaning
df = df[df["text"].str.len() > 0]

df["y"] = df["label"].map({"HUMAN": 0, "MACHINE": 1})

print("After cleaning:")
print(df["label"].value_counts())


After cleaning:
label
HUMAN      91269
MACHINE    29884
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split

X = df["text"].values
y = df["y"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", len(X_train))
print("Test :", len(X_test))


Train: 96922
Test : 24231


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

word_v = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=12000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    lowercase=True
)

Xtr_word = word_v.fit_transform(X_train)
Xte_word = word_v.transform(X_test)

print("Word features:", Xtr_word.shape)


Word features: (96922, 12000)


In [17]:
char_v = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=6000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

Xtr_char = char_v.fit_transform(X_train)
Xte_char = char_v.transform(X_test)

print("Char features:", Xtr_char.shape)


Char features: (96922, 6000)


In [18]:
from scipy.sparse import hstack

Xtr = hstack([Xtr_word, Xtr_char])
Xte = hstack([Xte_word, Xte_char])

print("Total features:", Xtr.shape[1])


Total features: 18000


In [19]:
from scipy.sparse import hstack

Xtr = hstack([Xtr_word, Xtr_char])
Xte = hstack([Xte_word, Xte_char])

print("Total features:", Xtr.shape[1])


Total features: 18000


In [21]:
from sklearn.linear_model import LogisticRegression

base_clf = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

base_clf.fit(Xtr, y_train)

print("✓ base_clf trained")


✓ base_clf trained


In [22]:
from sklearn.calibration import CalibratedClassifierCV

clf = CalibratedClassifierCV(
    base_clf,
    method="sigmoid",
    cv=5,
    n_jobs=-1
)

clf.fit(Xtr, y_train)

print("✓ Calibrated classifier ready")


✓ Calibrated classifier ready


In [23]:
print("base_clf exists:", "base_clf" in globals())
print("clf exists     :", "clf" in globals())
print("Xtr shape      :", Xtr.shape)
print("y_train shape  :", y_train.shape)


base_clf exists: True
clf exists     : True
Xtr shape      : (96922, 18000)
y_train shape  : (96922,)


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = clf.predict(Xte)

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["HUMAN", "MACHINE"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

       HUMAN       0.95      0.96      0.95     18254
     MACHINE       0.87      0.83      0.85      5977

    accuracy                           0.93     24231
   macro avg       0.91      0.90      0.90     24231
weighted avg       0.93      0.93      0.93     24231

Confusion Matrix:
[[17538   716]
 [  998  4979]]


In [25]:
import time
from scipy.sparse import hstack

tests = [
    "hello",
    "hi speaking",
    "yes hello",
    "please leave a message after the beep",
    "you have reached the voicemail of"
]

for t in tests:
    X = hstack([
        word_v.transform([t]),
        char_v.transform([t])
    ])

    start = time.time()
    pred = clf.predict(X)[0]
    proba = clf.predict_proba(X)[0]
    ms = (time.time() - start) * 1000

    decision = "MACHINE" if pred == 1 else "HUMAN"
    confidence = proba[pred]

    print(
        f"{t:45s} → {decision:7s} "
        f"conf={confidence:.3f} "
        f"{ms:.3f}ms"
    )


hello                                         → HUMAN   conf=1.000 3.770ms
hi speaking                                   → HUMAN   conf=0.973 5.209ms
yes hello                                     → HUMAN   conf=1.000 1.596ms
please leave a message after the beep         → MACHINE conf=0.997 1.596ms
you have reached the voicemail of             → MACHINE conf=0.996 1.590ms


In [27]:
import joblib
import os

os.makedirs("models", exist_ok=True)

MODEL_NAME = "text_cls_amd_v2.joblib"
MODEL_PATH = f"models/{MODEL_NAME}"

joblib.dump(bundle, MODEL_PATH)

print("✅ Model saved as:")
print(MODEL_PATH)


✅ Model saved as:
models/text_cls_amd_v2.joblib


In [29]:
import joblib
from scipy.sparse import hstack

b = joblib.load("models/text_cls_amd_v2.joblib")

X = hstack([
    b["word_v"].transform(["hello"]),
    b["char_v"].transform(["hello"])
])

print("Decision:", "MACHINE" if b["clf"].predict(X)[0] else "HUMAN")
print("Probabilities:", b["clf"].predict_proba(X))


Decision: HUMAN
Probabilities: [[9.99995016e-01 4.98441459e-06]]


In [30]:
import joblib
import time
from scipy.sparse import hstack

MODEL_PATH = "models/text_cls_amd_v2.joblib"

bundle = joblib.load(MODEL_PATH)

clf = bundle["clf"]
word_v = bundle["word_v"]
char_v = bundle["char_v"]

print("✓ Model loaded:", MODEL_PATH)


✓ Model loaded: models/text_cls_amd_v2.joblib


In [31]:
def test_text(text: str):
    if not text or not text.strip():
        return {
            "decision": "HUMAN",
            "confidence": 0.5,
            "reason": "empty_text",
            "latency_ms": 0.0
        }

    X = hstack([
        word_v.transform([text]),
        char_v.transform([text])
    ])

    start = time.time()
    pred = clf.predict(X)[0]
    proba = clf.predict_proba(X)[0]
    latency_ms = (time.time() - start) * 1000

    decision = "MACHINE" if pred == 1 else "HUMAN"
    confidence = float(proba[pred])

    return {
        "text": text,
        "decision": decision,
        "confidence": confidence,
        "proba_human": float(proba[0]),
        "proba_machine": float(proba[1]),
        "latency_ms": latency_ms
    }


In [32]:
print("\nType text to test the AMD classifier.")
print("Type 'exit' or 'quit' to stop.\n")

while True:
    text = input("You > ").strip()

    if text.lower() in ("exit", "quit"):
        print("Exiting tester.")
        break

    result = test_text(text)

    print(
        f"\nDecision : {result['decision']}\n"
        f"Confidence: {result['confidence']:.4f}\n"
        f"Proba(H)  : {result['proba_human']:.4f}\n"
        f"Proba(M)  : {result['proba_machine']:.4f}\n"
        f"Latency  : {result['latency_ms']:.3f} ms\n"
        + "-"*40
    )



Type text to test the AMD classifier.
Type 'exit' or 'quit' to stop.



You >  hello 



Decision : HUMAN
Confidence: 1.0000
Proba(H)  : 1.0000
Proba(M)  : 0.0000
Latency  : 1.766 ms
----------------------------------------


You >  leave a message



Decision : MACHINE
Confidence: 0.9939
Proba(H)  : 0.0061
Proba(M)  : 0.9939
Latency  : 1.728 ms
----------------------------------------


You >  press 1


... (21 outputs left out)

You >  Hello 



Decision : HUMAN
Confidence: 1.0000
Proba(H)  : 1.0000
Proba(M)  : 0.0000
Latency  : 1.721 ms
----------------------------------------


You >  exit


Exiting tester.


In [33]:
import joblib

b = joblib.load("models/text_cls_amd_v2.joblib")

print("Word TF-IDF features:", len(b["word_v"].vocabulary_))
print("Char TF-IDF features:", len(b["char_v"].vocabulary_))
print("Total features       :", 
      len(b["word_v"].vocabulary_) + len(b["char_v"].vocabulary_))


Word TF-IDF features: 12000
Char TF-IDF features: 6000
Total features       : 18000
